# SOXX series N-CSR/N-CSRS 检索页

两份固定 SEC 搜索页补足 N-Q/NPORT-P 以外的系列申报索引；修订形式另列，不读取正文和收益。


In [1]:
import json,hashlib,re,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from bs4 import BeautifulSoup
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_series_filing_search_20261023';plan=json.loads((card/'additional_forms_plan.json').read_text())
assert plan['frozen'] and not (card/'additional_forms_attempt_started.json').exists()
assert json.loads((card/plan['parent_result']).read_text())['status']==plan['parent_status']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=root/'data/snapshots/stage1_soxx_series_filing_search_20261023/additional_forms';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'additional_forms_attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];rows=[];error=None
for i,spec in enumerate(plan['requests_in_order']):
 state['GET_attempts']+=1;(card/'additional_forms_attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 name=spec['form'].replace('-','_');body=raw/(name+'.html');headers=raw/(name+'.headers');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(plan['transport']['connect_timeout_seconds']),'--max-time',str(plan['transport']['total_timeout_seconds']),'--max-filesize',str(plan['transport']['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['transport']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=60);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'form':spec['form'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:
  receipt={'form':spec['form'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:200]};status=None
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 receipts.append(receipt)
 if status!=200 or receipt.get('curl_exit_code')!=0 or not body.exists():error='transport_failure';break
 try:
  soup=BeautifulSoup(body.read_bytes(),'html.parser');identity=soup.select_one('p.identInfo')
  assert identity is not None and 'S000004354' in identity.get_text(' ',strip=True) and 'Semiconductor ETF' in identity.get_text(' ',strip=True)
  table=soup.select_one('#seriesDiv table.tableFile2');assert table is not None
  selected=[]
  for tr in table.select('tr'):
   cells=tr.find_all('td',recursive=False)
   if not cells:continue
   assert len(cells)==5
   form=cells[0].get_text(' ',strip=True);assert form in plan['accepted_form_variants'][spec['form']]
   link=cells[1].find('a',id='documentsbutton');assert link and link.get('href')
   match=re.search(r'([0-9]{10}-[0-9]{2}-[0-9]{6})-index\.htm',link['href']);assert match
   filed=cells[3].get_text(' ',strip=True);assert re.fullmatch(r'20[0-9]{2}-[0-9]{2}-[0-9]{2}',filed)
   selected.append({'form':form,'accession':match.group(1),'filed':filed,'index_href':link['href']})
  receipt['series_rows']=len(selected);receipt['first_filed']=min((x['filed'] for x in selected),default=None);receipt['last_filed']=max((x['filed'] for x in selected),default=None)
  rows.extend(selected)
 except Exception as exc:
  error='series_or_schema_failure:'+type(exc).__name__+':'+str(exc)[:180];receipt['parse_error']=error;break
 if i+1<len(plan['requests_in_order']):time.sleep(plan['transport']['seconds_between_requests'])
(card/'additional_forms_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
(card/'additional_forms_index.json').write_text(json.dumps({'series_id':'S000004354','rows':rows},indent=2)+'\n')
manifest={'plan_sha256':sha(card/'additional_forms_plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'receipts_sha256':sha(card/'additional_forms_receipts.json'),'index_sha256':sha(card/'additional_forms_index.json')}
(card/'additional_forms_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-additional-series-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ['additional_forms_manifest.json','additional_forms_receipts.json','additional_forms_index.json']:out.add(card/name,arcname=name)
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-additional-readback-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:
   run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
primary=json.loads((root/'research/stage1_soxx_legacy_basket_history_20261023/candidate_trust_filings.json').read_text())['rows'];primary_ids={x['accessionNumber'] for x in primary}
selected_2011_2018=[x for x in rows if '2011'<=x['filed'][:4]<='2018' and x['form'] in {'N-CSR','N-CSRS'}]
result={'status':'additional_series_shareholder_report_index_visible_source_only' if error is None and len(receipts)==2 else 'additional_series_index_partial_or_unqualified','error':error,'GET_attempts':state['GET_attempts'],'form_counts':{f:sum(x['form']==f for x in rows) for f in ['N-CSR','N-CSR/A','N-CSRS','N-CSRS/A']},'2011_2018_base_rows':len(selected_2011_2018),'2011_2018_base_in_trust_index':all(x['accession'] in primary_ids for x in selected_2011_2018),'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'holdings_values_qualified':False}
(card/'additional_forms_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','GET_attempts','form_counts','2011_2018_base_rows','2011_2018_base_in_trust_index','nas_archive_sha256')})


{'status': 'additional_series_index_partial_or_unqualified', 'GET_attempts': 1, 'form_counts': {'N-CSR': 0, 'N-CSR/A': 0, 'N-CSRS': 0, 'N-CSRS/A': 0}, '2011_2018_base_rows': 0, '2011_2018_base_in_trust_index': True, 'nas_archive_sha256': 'c22146476851d3c9853d9c1da489fe54402f113693f72e56e176d98111f5da71'}


## 中文结论

固定 N-CSR 请求页 HTTP 200 且原字节 NAS 归档；SEC 前缀检索实际合并 N-CSR、N-CSRS 和修订形式，初版单一表格断言失败并保存回执。新零网络修复另见 additional_forms_parse_repair.ipynb；按修复计划没有发第二次重复 GET。未读取正文、收益或模型。
